# [3단계 GeoAI] 항공 25cm 파인튜닝 — YOLO (재시도 2차)

0.5m 위성에서 제로샷·약지도 모두 실패 → **25cm 항공**(해상도 4배)으로 재시도.
25cm에선 철탑의 격자 구조·전선까지 육안 식별 확인됨(크롭 검증).

- **학습 권역**: 화성·세종 / **평가 권역**: 서천·보령·은평 (OSM→OSM 순환 차단)
- 데이터는 설비 중심 1024px 크롭 + 동수 배경, HF `yolo25_dataset.zip` 하나.
- 상자: 철탑 48px(12m)·변전소 192px(48m) — 0.5m 때의 2배(해상도 비례).
- **Kaggle: Accelerator=GPU T4, Internet=On** 필수.

## 순서
1. 셀 2에 `HF_TOKEN`·`HF_REPO` 입력 → Run All.
2. 학습(~1시간) → 평가 지표 → 가중치·로그를 HF `yolo25_out/`으로 반출.
3. 로컬에서 best.pt 회수 후 점-거리 OSM 대조(엄밀 평가).

In [ ]:
# 셀 1 — 설치
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음!')
%pip -q install ultralytics huggingface_hub --upgrade


In [ ]:
# 셀 2 — 설정 + 데이터셋 반입 (항공 25cm 크롭)
HF_TOKEN = ''
HF_REPO  = ''   # 예 'jyxnq/wp2026-tiles-pilot'
EPOCHS, IMGSZ, BATCH = 80, 1024, 8

import os, zipfile
BASE = '/content' if os.path.isdir('/content') else '/kaggle/working'
DS = os.path.join(BASE, 'yolo25')
from huggingface_hub import hf_hub_download
z = hf_hub_download(repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN or None,
                    filename='yolo25_dataset.zip')
os.makedirs(DS, exist_ok=True)
zipfile.ZipFile(z).extractall(DS)
# dataset.yaml의 path를 절대경로로 고정
yml = os.path.join(DS, 'dataset.yaml')
txt = open(yml, encoding='utf-8').read().replace('path: .', f'path: {DS}')
open(yml, 'w', encoding='utf-8').write(txt)
print(open(yml, encoding='utf-8').read())
print('train 이미지', len(os.listdir(os.path.join(DS, 'train/images'))),
      '· val 이미지', len(os.listdir(os.path.join(DS, 'val/images'))))

In [ ]:
# 셀 3 — 학습 (YOLOv8s, 작은 객체라 imgsz=1024 유지)
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data=os.path.join(DS, 'dataset.yaml'), epochs=EPOCHS, imgsz=IMGSZ,
            batch=BATCH, patience=20, project=os.path.join(BASE, 'runs'),
            name='pylon25', exist_ok=True,
            # 작은 객체·약지도: 과한 증강 억제
            mosaic=0.5, scale=0.3, fliplr=0.5, degrees=0.0, translate=0.1)
print('학습 완료')

In [ ]:
# 셀 4 — 평가 권역 val 세트 지표 (YOLO 내장 — OSM 약라벨 기준 mAP/P/R)
best = os.path.join(BASE, 'runs', 'pylon25', 'weights', 'best.pt')
m = YOLO(best)
metrics = m.val(data=os.path.join(DS, 'dataset.yaml'), imgsz=IMGSZ, split='val')
print('val mAP50:', round(float(metrics.box.map50), 3))
print('클래스별 P/R/mAP50:')
for i, name in enumerate(['tower', 'substation']):
    try:
        print(f'  {name}: P={metrics.box.p[i]:.3f} R={metrics.box.r[i]:.3f} '
              f'mAP50={metrics.box.ap50[i]:.3f}')
    except Exception:
        pass

In [ ]:
# 셀 5 — 가중치·학습로그를 HF로 반출 (로컬 회수용)
if HF_TOKEN and HF_REPO:
    from huggingface_hub import upload_file
    runs = os.path.join(BASE, 'runs', 'pylon25')
    for rel in ['weights/best.pt', 'results.csv', 'confusion_matrix.png',
                'val_batch0_pred.jpg']:
        p = os.path.join(runs, rel)
        if os.path.isfile(p):
            upload_file(path_or_fileobj=p, path_in_repo=f'yolo25_out/{os.path.basename(rel)}',
                        repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN)
            print('업로드:', rel)
    print('완료 — 로컬에서 best.pt 회수 후 점-거리 OSM 대조')

## 다음 (로컬)
1. HF `yolo25_out/best.pt` 회수.
2. 평가 권역(서천·보령·은평) 도엽에 추론 → 도엽 아핀(s3_sheet_georef.csv)으로
   EPSG:5186→5179 좌표화 → `s3_detect_vs_osm.py`로 점-거리 OSM 대조(철탑 50m).
   셀 4의 mAP는 약라벨 상자 기준 참고치이고, 본 평가는 로컬에서 한다.
3. 판정 기준: 0.5m 결과(제로샷 정밀도 0.9%·약지도 탐지 0건) 대비 개선 여부.
   25cm에서도 실패하면 GeoAI 트랙 최종 기각, 한전 목록 표-대-표 대조로 확정.